# Graded Retraining on the Full Forget Set — Notebook 2 of 10

Re-evaluates the graded retrain checkpoints (`ret25`, `ret50`, `ret75` in `Retrain Models/`) on the
original forget/retain split and overwrites the `retrain25/50/75` result files in `Results_Fixed/Results_<name>/`.

**Jobs in this notebook:** LiRA seed 1 (ret50); MUCAC seeds 3–4; Full class (all seeds)

**Estimated run time:** about 14.3 h (plus setup)

Run all cells. The last cell creates `compiled_results.csv` and `compiled_results_MIAU.csv` only for
experiments whose graded files have all been re-evaluated; run it again later in any notebook to
pick up experiments finished by other notebooks. Then run `postprocess.py`.

In [ ]:
# ============ CONFIGURE THIS ONCE ============
BASE_PATH = "/content/drive/MyDrive/DeepUnlearning/Unlearning/source_submitted"
# =============================================

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir(BASE_PATH)
print(f"Working directory: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")

In [ ]:
# Uninstall default transformers/tokenizers (version mismatch with ViT checkpoints)
!pip uninstall -y transformers tokenizers

In [ ]:
# Install Rust (needed to compile tokenizers) and specific transformers version for ViT compatibility
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

import os
os.environ["PATH"] += ":/root/.cargo/bin"
# Newer Rust rejects some code in old tokenizers; this flag allows it
os.environ["RUSTFLAGS"] = "-A invalid_reference_casting"

!pip install tokenizers==0.13.3 transformers==4.30.0

In [ ]:
# Install dependencies (transformers installed separately above with specific version)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install xgboost scikit-learn pandas numpy matplotlib tqdm

In [ ]:
!pip install torchmetrics

In [ ]:
# Imports and setup
import json
import os
import subprocess
import sys
import os
os.environ['PYTHONUNBUFFERED'] = '1'

sys.path.insert(0, BASE_PATH)

# Load configuration
with open(os.path.join(BASE_PATH, 'config.json'), 'r') as f:
    CONFIG = json.load(f)

SEEDS = CONFIG['seeds']
OUTPUT_BASE = CONFIG['output_base']
# Use method_configs which includes ret_perc for retrain25/50/75 experiments
METHOD_CONFIGS = CONFIG.get('method_configs', [
    {"name": m, "method": m, "ret_perc": 0} for m in CONFIG.get('methods', [])
])

# Create output directory
os.makedirs(OUTPUT_BASE, exist_ok=True)
print(f"Output directory: {OUTPUT_BASE}")
print(f"Seeds: {SEEDS}")
print(f"Method configs: {[m['name'] for m in METHOD_CONFIGS]}")
print(f"Number of experiments: {len(CONFIG['experiments'])}")

In [ ]:
# ========== HELPER FUNCTIONS ==========

def run_command_realtime(cmd):
    """
    Run a command with real-time output streaming.
    Uses Popen to stream stdout line by line.
    """
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    return process.returncode


def resolve_retrain_folder(rel_or_abs):
    """
    Locate a retrain-model folder. Tries BASE_PATH first, then its parent
    (where Retrain Models often lives next to source_submitted).
    """
    if not rel_or_abs:
        return None
    if os.path.isabs(rel_or_abs) and os.path.exists(rel_or_abs):
        return rel_or_abs
    candidates = [
        os.path.join(BASE_PATH, rel_or_abs),
        os.path.join(os.path.dirname(BASE_PATH.rstrip("/")), rel_or_abs),
        rel_or_abs,
    ]
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return candidates[0]


def run_main_py(exp_config, seed, method_config):
    """
    Run main.py with the given configuration.
    
    method_config: dict with 'name', 'method', 'ret_perc' keys
        e.g., {"name": "retrain25", "method": "retrain", "ret_perc": 25}
    
    Returns: 0 if skipped (already exists), returncode otherwise
    """
    output_dir = os.path.join(OUTPUT_BASE, f"Results_{exp_config['name']}")
    os.makedirs(output_dir, exist_ok=True)
    
    # Support both old-style string method and new-style dict method_config
    if isinstance(method_config, str):
        method_name = method_config
        method_cmd = method_config
        ret_perc = 0
    else:
        method_name = method_config['name']
        method_cmd = method_config['method']
        ret_perc = method_config.get('ret_perc', 0)
    
    # Check if result file already exists (resume capability)
    output_file = f"{method_name}_{exp_config['dataset'].lower()}_{exp_config['model'].lower()}_seed_{seed}.txt"
    output_path = os.path.join(output_dir, output_file)
    if os.path.exists(output_path):
        print(f"⏭️ Skipping {output_file} - already exists")
        return 0
    
    cmd = [
        "python", "-u", "main.py",
        "-net", exp_config['model'],
        "-dataset", exp_config['dataset'],
        "-classes", str(exp_config['num_classes']),
        "-weight_path", os.path.join(BASE_PATH, exp_config['baseline_path']),
        "-csv_path", os.path.join(BASE_PATH, exp_config['split_csv']),
        "-forget_per_class", str(exp_config['forget_per_class']),
        "-method", method_cmd,
        "-ret_perc", str(ret_perc),
        "-seed", str(seed),
        "-b", str(exp_config.get('batch_size', 256)),
        "-output_dir", output_dir,
        "-gpu",
    ]
    
    retrain_folder = resolve_retrain_folder(exp_config.get("retrain_folder"))
    if retrain_folder:
        cmd.extend(["-retrain_folder", retrain_folder])
    
    # Add augmentation flag if needed
    if not exp_config.get('use_augmentation', True):
        cmd.append("-no_augmentation")
    
    # Add epochs override for underfitted
    if 'epochs_retrain' in exp_config:
        cmd.extend(["-epochs_override", str(exp_config['epochs_retrain'])])
    if 'milestones' in exp_config:
        cmd.extend(["-milestones_override", ','.join(map(str, exp_config['milestones']))])
    
    print(f"\n{'='*60}")
    print(f"Running: {exp_config['name']} | Seed: {seed} | Method: {method_name}")
    print(f"Command: {' '.join(cmd)}")
    print(f"{'='*60}", flush=True)
    
    return run_command_realtime(cmd)


def run_train_model(model_name, dataset_name, num_classes, epochs, milestones,
                    use_augmentation, output_path, batch_size=256, seed=None):
    """
    Run train_model.py to train a baseline model.
    """
    cmd = [
        "python", "-u", "train_model.py",
        "-net", model_name,
        "-dataset", dataset_name,
        "-classes", str(num_classes),
        "-epochs_override", str(epochs),
        "-milestones_override", ','.join(map(str, milestones)),
        "-b", str(batch_size),
        "-output_path", output_path,
        "-gpu",
    ]
    
    if not use_augmentation:
        cmd.append("-no_augmentation")
    
    if seed is not None:
        cmd.extend(["-seed", str(seed)])
    
    print(f"\n{'='*60}")
    print(f"Training baseline: {model_name} on {dataset_name}")
    print(f"Epochs: {epochs}, Augmentation: {use_augmentation}")
    print(f"Command: {' '.join(cmd)}")
    print(f"{'='*60}", flush=True)
    
    return run_command_realtime(cmd)


def run_convert_csv(folder_path):
    """
    Run convert_csv.py to compile results.
    """
    cmd = ["python", "-u", "convert_csv.py", "-folder", folder_path]
    
    print(f"\nCompiling results in: {folder_path}", flush=True)
    run_command_realtime(cmd)
    
    return os.path.join(folder_path, 'compiled_results.csv')


def run_calculate_miau(csv_path):
    """
    Run calculate_miau_per_seed.py on a compiled CSV.
    """
    cmd = ["python", "-u", "calculate_miau_per_seed.py", "--input", csv_path]
    
    print(f"\nCalculating MIAU for: {csv_path}", flush=True)
    return run_command_realtime(cmd)


def run_main_new_metrics(exp_config, seed, method_config, attack="standard"):
    """
    Run main_new_metrics.py for unlearning with advanced MIA metrics.
    
    method_config: dict with 'name', 'method', 'ret_perc' keys
    attack: MIA attack type ('standard', 'xgboost', 'lira', 'quantile', 'shadow')
    
    Returns: 0 if skipped (already exists), returncode otherwise
    """
    # Create output directory based on attack type
    output_dir = os.path.join(OUTPUT_BASE, f"Results_{exp_config['name']}_{attack.capitalize()}")
    os.makedirs(output_dir, exist_ok=True)
    
    # Support both old-style string and new-style dict
    if isinstance(method_config, str):
        method_name = method_config
        method_cmd = method_config
        ret_perc = 0
    else:
        method_name = method_config['name']
        method_cmd = method_config['method']
        ret_perc = method_config.get('ret_perc', 0)
    
    # Check if result file already exists (resume capability)
    output_file = f"{method_name}_{exp_config['dataset'].lower()}_{exp_config['model'].lower()}_seed_{seed}.txt"
    output_path = os.path.join(output_dir, output_file)
    if os.path.exists(output_path):
        print(f"⏭️ Skipping {output_file} - already exists")
        return 0
    
    cmd = [
        "python", "-u", "main_new_metrics.py",
        "-net", exp_config['model'],
        "-dataset", exp_config['dataset'],
        "-classes", str(exp_config['num_classes']),
        "-weight_path", os.path.join(BASE_PATH, exp_config['baseline_path']),
        "-csv_path", os.path.join(BASE_PATH, exp_config['split_csv']),
        "-forget_per_class", str(exp_config['forget_per_class']),
        "-method", method_cmd,
        "-ret_perc", str(ret_perc),
        "-seed", str(seed),
        "-b", str(exp_config.get('batch_size', 256)),
        "-output_dir", output_dir,
        "-attack", attack,
        "-gpu",
    ]
    
    retrain_folder = resolve_retrain_folder(exp_config.get("retrain_folder"))
    if retrain_folder:
        cmd.extend(["-retrain_folder", retrain_folder])
    
    # Add augmentation flag if needed
    if not exp_config.get('use_augmentation', True):
        cmd.append("-no_augmentation")
    
    print(f"\n{'='*60}")
    print(f"Running {attack.upper()} MIA: {method_name} | Seed: {seed}")
    print(f"Command: {' '.join(cmd)}")
    print(f"{'='*60}", flush=True)
    
    return run_command_realtime(cmd)

In [ ]:
import zipfile
import os

# Unzip CelebAMask-HQ to local storage
zip_path = "/content/drive/MyDrive/DeepUnlearning/Unlearning/source_submitted/data/MUCAC/CelebAMask-HQ.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("/content/")

# Check what was extracted
print(os.listdir("/content/"))

os.rename("/content/CelebAMask-HQ", "/content/MUCAC")


In [ ]:
import os
import shutil

# 1. Create CelebAMask-HQ subfolder
os.makedirs("/content/MUCAC/CelebAMask-HQ", exist_ok=True)

# 2. Move all current contents into CelebAMask-HQ subfolder
for item in os.listdir("/content/MUCAC"):
    if item != "CelebAMask-HQ":  # Don't move the folder into itself
        src = f"/content/MUCAC/{item}"
        dst = f"/content/MUCAC/CelebAMask-HQ/{item}"
        shutil.move(src, dst)
        print(f"Moved: {item}")

# 3. Copy the two additional files from Drive
drive_path = "/content/drive/MyDrive/DeepUnlearning/Unlearning/source_submitted/data/MUCAC"

shutil.copy(f"{drive_path}/CelebA-HQ-attribute.txt", "/content/MUCAC/")
print("Copied: CelebA-HQ-attribute.txt")

shutil.copy(f"{drive_path}/CelebA-HQ-identity.txt", "/content/MUCAC/")
print("Copied: CelebA-HQ-identity.txt")

# Verify final structure
print("\n/content/MUCAC/ contents:")
for item in os.listdir("/content/MUCAC"):
    print(f"  {item}")

In [ ]:
# ========== LOCAL WORKING DIRECTORY ==========
# LiRA/Shadow/Quantile write original_indices_* files into the working directory,
# so each notebook needs its own when several run against the same Drive folder.
# Results are still written to Results_Fixed on Drive.
import glob
import shutil

WORK_DIR = "/content/work"
os.makedirs(WORK_DIR, exist_ok=True)
for p in glob.glob(os.path.join(BASE_PATH, "*.py")) + [os.path.join(BASE_PATH, "config.json")]:
    shutil.copy2(p, WORK_DIR)
for d in ["dualoptim", "lira", "quantile", "shadow_attack", "splits"]:
    shutil.copytree(os.path.join(BASE_PATH, d), os.path.join(WORK_DIR, d), dirs_exist_ok=True)
if not os.path.exists(os.path.join(WORK_DIR, "data")):
    os.symlink(os.path.join(BASE_PATH, "data"), os.path.join(WORK_DIR, "data"))

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
OUTPUT_BASE = os.path.join(BASE_PATH, CONFIG['output_base'])
print(f"Working directory: {os.getcwd()}")
print(f"Results go to: {OUTPUT_BASE}")

In [ ]:
# ========== GRADED RETRAINS RE-EVALUATED ON THE FULL FORGET SET ==========
# Each graded checkpoint is scored with -method baseline -ret_perc 0, i.e. on the
# original forget/retain split, and overwrites the existing retrain25/50/75 files
# in Results_<name> (same names and labels as before, so postprocess.py runs unchanged).
# Files written here carry a marker line so a restart skips finished runs.
import re
from retrain_utils import find_best_retrain_model

GE = CONFIG['graded_eval']
RET_PERCS = GE.get('ret_percs', [25, 50, 75])
MARKER = "Evaluation: full forget set"

EXP_BY_NAME = {e['name']: ("standard", e) for e in CONFIG['experiments']}
EXP_BY_NAME.update({e['name']: ("fullclass", e) for e in CONFIG.get('fullclass_experiments', [])})
EXP_BY_NAME.update({e['name']: ("subclass", e) for e in CONFIG.get('subclass_experiments', [])})
EXP_BY_NAME.update({e['name']: ("advanced", e) for e in CONFIG.get('advanced_mia_experiments', [])})


def find_graded_ckpt(folder, exp_config, seed, ret_perc):
    # find_best_retrain_model falls back to any ret level if the requested one is missing
    path = find_best_retrain_model(folder, exp_config['model'], exp_config['dataset'], seed, ret_perc=ret_perc)
    if path is None:
        return None
    m = re.search(r"-ret(\d+)-\d+-best\.pth$", os.path.basename(path))
    return path if m and int(m.group(1)) == ret_perc else None


def is_full_forget(txt_path):
    if not os.path.exists(txt_path):
        return False
    with open(txt_path) as f:
        return MARKER in f.read()


def finalize(txt_path, label, ckpt):
    with open(txt_path) as f:
        content = f.read()
    content = re.sub(r"^unlearning:.*$", f"unlearning: {label}", content, count=1, flags=re.MULTILINE)
    content = content.rstrip("\n") + f"\n{MARKER}\nCheckpoint: {os.path.basename(ckpt)}\n"
    with open(txt_path, 'w') as f:
        f.write(content)


def build_cmd(kind, exp_config, seed, ckpt, out_dir, out_file, attack=None):
    tail = ["-method", "baseline", "-ret_perc", "0", "-seed", str(seed),
            "-output_dir", out_dir, "-output_file", out_file, "-gpu"]
    if kind in ("standard", "advanced"):
        cmd = ["python", "-u", "main.py" if kind == "standard" else "main_new_metrics.py",
               "-net", exp_config['model'], "-dataset", exp_config['dataset'],
               "-classes", str(exp_config['num_classes']), "-weight_path", ckpt,
               "-csv_path", os.path.join(BASE_PATH, exp_config['split_csv']),
               "-forget_per_class", str(exp_config['forget_per_class']),
               "-b", str(exp_config.get('batch_size', 256))] + tail
        if not exp_config.get('use_augmentation', True):
            cmd.append("-no_augmentation")
        if kind == "advanced":
            cmd += ["-attack", attack]
    else:
        cmd = ["python", "-u", "forget_full_class_main.py" if kind == "fullclass" else "forget_subclass_main.py",
               "-net", exp_config['model'], "-weight_path", ckpt,
               "-dataset", exp_config['dataset'], "-classes", str(exp_config['num_classes']),
               "-forget_class", exp_config['forget_class']] + tail
        if kind == "subclass":
            cmd += ["-my_seed", str(seed)]
    return cmd


def run_graded_eval(kind, exp_config, seeds, attack=None, ret_percs=None):
    name = exp_config['name']
    folder = resolve_retrain_folder(GE['folders'].get(name))
    if not folder:
        print(f"Skipping {name}: graded checkpoint folder not found")
        return
    tag = f"{name}_{attack.capitalize()}" if attack else name
    output_dir = os.path.join(OUTPUT_BASE, f"Results_{tag}")
    os.makedirs(output_dir, exist_ok=True)
    print(f"\n{'#'*80}\n# GRADED, FULL FORGET SET: {tag} | seeds {seeds}\n# Checkpoints: {folder}\n{'#'*80}", flush=True)

    for seed in seeds:
        for ret_perc in (ret_percs or RET_PERCS):
            label = f"retrain{ret_perc}"
            output_file = f"{label}_{exp_config['dataset'].lower()}_{exp_config['model'].lower()}_seed_{seed}.txt"
            output_path = os.path.join(output_dir, output_file)
            if is_full_forget(output_path):
                print(f"⏭️ Skipping {output_file} - already re-evaluated")
                continue
            ckpt = find_graded_ckpt(folder, exp_config, seed, ret_perc)
            if ckpt is None:
                print(f"⚠️ No ret{ret_perc} checkpoint for seed {seed} in {folder} - keeping existing file")
                continue
            cmd = build_cmd(kind, exp_config, seed, ckpt, output_dir, output_file, attack)
            print(f"\n{label} | seed {seed}\n{' '.join(cmd)}", flush=True)
            if run_command_realtime(cmd) == 0:
                finalize(output_path, label, ckpt)


def run_jobs(jobs):
    for job in jobs:
        kind, exp_config = EXP_BY_NAME[job['exp']]
        run_graded_eval(kind, exp_config, job.get('seeds', SEEDS),
                        attack=job.get('attack'), ret_percs=job.get('ret_percs'))

In [ ]:
# ========== JOBS FOR THIS NOTEBOOK ==========
ADV = "ResNet18_Cifar10_AdvancedMIA"

NOTEBOOK_JOBS = {
    1: [{"exp": ADV, "attack": "lira", "seeds": [1], "ret_percs": [25]},
        {"exp": "ResNet18_MUCAC", "seeds": [1, 2]},
        {"exp": "ViT_Cifar10", "seeds": [5, 6]}],
    2: [{"exp": ADV, "attack": "lira", "seeds": [1], "ret_percs": [50]},
        {"exp": "ResNet18_MUCAC", "seeds": [3, 4]},
        {"exp": "AllCNN_Cifar20_FullClass"}],
    3: [{"exp": ADV, "attack": "lira", "seeds": [1], "ret_percs": [75]},
        {"exp": "ResNet18_MUCAC", "seeds": [5, 6]},
        {"exp": "AllCNN_Cifar20_SubClass"}],
    4: [{"exp": ADV, "attack": "lira", "seeds": [2], "ret_percs": [25]},
        {"exp": "ResNet18_MUCAC", "seeds": [7, 8]},
        {"exp": "ResNet18_Cifar10_Underfitted"}],
    5: [{"exp": ADV, "attack": "lira", "seeds": [2], "ret_percs": [50]},
        {"exp": "ResNet18_MUCAC", "seeds": [9, 10]},
        {"exp": "ResNet18_Cifar10_Overfitted"}],
    6: [{"exp": ADV, "attack": "lira", "seeds": [2], "ret_percs": [75]},
        {"exp": ADV, "attack": "xgboost", "seeds": [1, 2]},
        {"exp": ADV, "attack": "quantile", "seeds": [1, 2, 3]}],
    7: [{"exp": ADV, "attack": "lira", "seeds": [3], "ret_percs": [25]},
        {"exp": ADV, "attack": "xgboost", "seeds": [3, 4]},
        {"exp": ADV, "attack": "shadow", "seeds": [1, 2, 3]}],
    8: [{"exp": ADV, "attack": "lira", "seeds": [3], "ret_percs": [50]},
        {"exp": ADV, "attack": "xgboost", "seeds": [5, 6]},
        {"exp": "ViT_Cifar10", "seeds": [1, 2]}],
    9: [{"exp": ADV, "attack": "lira", "seeds": [3], "ret_percs": [75]},
        {"exp": ADV, "attack": "xgboost", "seeds": [7, 8]},
        {"exp": "ViT_Cifar10", "seeds": [3, 4]}],
    10: [{"exp": ADV, "attack": "xgboost", "seeds": [9, 10]},
         {"exp": "ViT_Cifar10", "seeds": [7, 8, 9, 10]},
         {"exp": "ResNet18_Cifar10"}, {"exp": "AllCNN_Cifar10"},
         {"exp": "ResNet18_Cifar20"}, {"exp": "AllCNN_Cifar20"},
         {"exp": "ResNet18_Mnist"}, {"exp": "AllCNN_Mnist"}],
}

NOTEBOOK_ID = 2
run_jobs(NOTEBOOK_JOBS[NOTEBOOK_ID])

In [ ]:
# ========== CREATE CSVs ONLY FOR EXPERIMENTS WHOSE GRADED FILES ARE ALL RE-EVALUATED ==========
# A graded file counts as done if it was written on or after 20 Oct 2026 and carries the
# full-forget marker. Folders with any missing or old file are skipped, so no CSV mixes
# old and new graded results. Run postprocess.py on the CSVs afterwards.
from datetime import datetime

CUTOFF = datetime(2026, 10, 20).timestamp()


def missing_graded_files(exp_config, seeds, output_dir):
    missing = []
    for seed in seeds:
        for ret_perc in RET_PERCS:
            name = f"retrain{ret_perc}_{exp_config['dataset'].lower()}_{exp_config['model'].lower()}_seed_{seed}.txt"
            path = os.path.join(output_dir, name)
            if not (is_full_forget(path) and os.path.getmtime(path) >= CUTOFF):
                missing.append(name)
    return missing


for name, (kind, exp_config) in EXP_BY_NAME.items():
    if name not in GE['folders']:
        continue
    attacks = exp_config.get('attacks', []) if kind == "advanced" else [None]
    for attack in attacks:
        tag = f"{name}_{attack.capitalize()}" if attack else name
        seeds = SEEDS[:3] if attack in ("lira", "quantile", "shadow") else SEEDS
        output_dir = os.path.join(OUTPUT_BASE, f"Results_{tag}")
        expected = 3 * len(seeds)
        missing = missing_graded_files(exp_config, seeds, output_dir)
        if missing:
            print(f"⏳ {tag}: {expected - len(missing)}/{expected} re-evaluated - CSV not created yet")
            continue
        print(f"✅ {tag}: all {expected} graded files re-evaluated - creating CSVs")
        csv_path = run_convert_csv(output_dir)
        if os.path.exists(csv_path):
            run_calculate_miau(csv_path)